In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [4]:
df_spark = spark.read\
           .option("header", "true")\
           .csv("Data/raw/green/2021/01")

In [5]:
df_spark.printSchema()

root
 |-- VendorID: string (nullable = true)
 |-- lpep_pickup_datetime: string (nullable = true)
 |-- lpep_dropoff_datetime: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- ehail_fee: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- trip_type: string (nullable = true)
 |-- congestion_surcharge: string (nullable = true)



In [6]:
df_spark.show(5)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2021-01-01 00:15:56|  2021-01-01 00:19:52|                 N|         1|          43|         151|              1|         1.01|        5.5|  0.5|    0.

In [7]:
import pandas as pd  
df_green_pd = pd.read_csv("Data/raw/green/2021/01/green_tripdata_2021_01.csv.gz", nrows=1000)

In [8]:
df_green_pd.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1,43,151,1,1.01,5.5,0.5,0.5,0.00,0.0,NaN,0.3,6.80,2,1,0.00
1,2,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1,166,239,1,2.53,10.0,0.5,0.5,2.81,0.0,NaN,0.3,16.86,1,1,2.75
2,2,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1,41,42,1,1.12,6.0,0.5,0.5,1.00,0.0,NaN,0.3,8.30,1,1,0.00
3,2,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1,168,75,1,1.99,8.0,0.5,0.5,0.00,0.0,NaN,0.3,9.30,2,1,0.00
4,2,2021-01-01 00:16:36,2021-01-01 00:16:40,N,2,265,265,3,0.00,-52.0,0.0,-0.5,0.00,0.0,NaN,-0.3,-52.80,3,1,0.00


In [9]:
df_green_pd.dtypes

VendorID                   int64
lpep_pickup_datetime         str
lpep_dropoff_datetime        str
store_and_fwd_flag           str
RatecodeID                 int64
PULocationID               int64
DOLocationID               int64
passenger_count            int64
trip_distance            float64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
ehail_fee                float64
improvement_surcharge    float64
total_amount             float64
payment_type               int64
trip_type                  int64
congestion_surcharge     float64
dtype: object

In [10]:
spark.createDataFrame(df_green_pd).schema

StructType([StructField('VendorID', LongType(), True), StructField('lpep_pickup_datetime', StringType(), True), StructField('lpep_dropoff_datetime', StringType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('RatecodeID', LongType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('ehail_fee', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('payment_type', LongType(), True), StructField('trip_type', LongType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [15]:
from pyspark.sql import types

green_schema = types.StructType(
    [
        types.StructField('VendorID', types.IntegerType(), True), 
        types.StructField('lpep_pickup_datetime', types.TimestampType(), True), 
        types.StructField('lpep_dropoff_datetime', types.TimestampType(), True), 
        types.StructField('store_and_fwd_flag', types.StringType(), True), 
        types.StructField('RatecodeID', types.IntegerType(), True), 
        types.StructField('PULocationID', types.IntegerType(), True), 
        types.StructField('DOLocationID', types.IntegerType(), True), 
        types.StructField('passenger_count', types.IntegerType(), True), 
        types.StructField('trip_distance', types.DoubleType(), True), 
        types.StructField('fare_amount', types.DoubleType(), True), 
        types.StructField('extra', types.DoubleType(), True), 
        types.StructField('mta_tax', types.DoubleType(), True), 
        types.StructField('tip_amount', types.DoubleType(), True), 
        types.StructField('tolls_amount', types.DoubleType(), True), 
        types.StructField('ehail_fee', types.DoubleType(), True), 
        types.StructField('improvement_surcharge', types.DoubleType(), True), 
        types.StructField('total_amount', types.DoubleType(), True), 
        types.StructField('payment_type', types.IntegerType(), True), 
        types.StructField('trip_type', types.IntegerType(), True), 
        types.StructField('congestion_surcharge', types.DoubleType(), True)
    ]
)

In [22]:
df_spark_green = spark.read\
           .option("header", "true")\
           .schema(green_schema)\
           .csv("Data/raw/green/2021/01")
df_spark_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [18]:
df_yellow_pd = pd.read_csv("Data/raw/yellow/2021/01/yellow_tripdata_2021_01.csv.gz", nrows=1000)
df_yellow_pd.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
0,1,2021-01-01 00:30:10,2021-01-01 00:36:12,1,2.10,1,N,142,43,2,8.0,3.0,0.5,0.00,0.0,0.3,11.80,2.5
1,1,2021-01-01 00:51:20,2021-01-01 00:52:19,1,0.20,1,N,238,151,2,3.0,0.5,0.5,0.00,0.0,0.3,4.30,0.0
2,1,2021-01-01 00:43:30,2021-01-01 01:11:06,1,14.70,1,N,132,165,1,42.0,0.5,0.5,8.65,0.0,0.3,51.95,0.0
3,1,2021-01-01 00:15:48,2021-01-01 00:31:01,0,10.60,1,N,138,132,1,29.0,0.5,0.5,6.05,0.0,0.3,36.35,0.0
4,2,2021-01-01 00:31:49,2021-01-01 00:48:21,1,4.94,1,N,68,33,1,16.5,0.5,0.5,4.06,0.0,0.3,24.36,2.5


In [19]:
df_yellow_pd.dtypes

VendorID                   int64
tpep_pickup_datetime         str
tpep_dropoff_datetime        str
passenger_count            int64
trip_distance            float64
RatecodeID                 int64
store_and_fwd_flag           str
PULocationID               int64
DOLocationID               int64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
congestion_surcharge     float64
dtype: object

In [20]:
spark.createDataFrame(df_yellow_pd).schema

StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', StringType(), True), StructField('tpep_dropoff_datetime', StringType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [21]:
yellow_schema = types.StructType(
                    [
                     types.StructField('VendorID', types.IntegerType(), True), 
                     types.StructField('tpep_pickup_datetime', types.TimestampType(), True), 
                     types.StructField('tpep_dropoff_datetime', types.TimestampType(), True), 
                     types.StructField('passenger_count', types.IntegerType(), True), 
                     types.StructField('trip_distance', types.DoubleType(), True), 
                     types.StructField('RatecodeID', types.IntegerType(), True), 
                     types.StructField('store_and_fwd_flag', types.StringType(), True), 
                     types.StructField('PULocationID', types.IntegerType(), True), 
                     types.StructField('DOLocationID', types.IntegerType(), True), 
                     types.StructField('payment_type', types.IntegerType(), True), 
                     types.StructField('fare_amount', types.DoubleType(), True), 
                     types.StructField('extra', types.DoubleType(), True), 
                     types.StructField('mta_tax', types.DoubleType(), True), 
                     types.StructField('tip_amount', types.DoubleType(), True), 
                     types.StructField('tolls_amount', types.DoubleType(), True), 
                     types.StructField('improvement_surcharge', types.DoubleType(), True), 
                     types.StructField('total_amount', types.DoubleType(), True), 
                     types.StructField('congestion_surcharge', types.DoubleType(), True)
                ]
)

In [23]:
df_spark_yellow = spark.read\
                  .option("header", "true")\
                  .schema(yellow_schema)\
                  .csv("Data/raw/yellow/2021/01")
df_spark_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



```python

df_spark_green = spark.read\
           .option("header", "true")\
           .schema(green_schema)\
           .csv("Data/raw/green/2021/01")
df_spark_green.printSchema() 

df_spark_green.repartition(4).write.parquet("Data/pq/green/2021/01")
```

In [25]:
tag = "green"

In [27]:
print(globals()[f"{tag}_schema"])

StructType([StructField('VendorID', IntegerType(), True), StructField('lpep_pickup_datetime', TimestampType(), True), StructField('lpep_dropoff_datetime', TimestampType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('RatecodeID', IntegerType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('passenger_count', IntegerType(), True), StructField('trip_distance', DoubleType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('ehail_fee', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('payment_type', IntegerType(), True), StructField('trip_type', IntegerType(), True), StructField('congestion_surcharge',

In [30]:
def write_to_parquet(months: int, tag: str = "green", year: int = 2021):
    """
    Reads CSV data and writes it to Parquet format.
    Skips months where input data does not exist.
    """

    schemas = {
        "green": green_schema,
        "yellow": yellow_schema
    }

    def path_exists(path: str) -> bool:
        fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
            spark._jsc.hadoopConfiguration()
        )
        return fs.exists(spark._jvm.org.apache.hadoop.fs.Path(path))

    for month in range(1, months + 1):

        input_path = f"Data/raw/{tag}/{year}/{month:02d}"
        output_path = f"Data/pq/{tag}/{year}/{month:02d}"

        # ✅ Check if path exists
        if not path_exists(input_path):
            print(f"Skipping {input_path} (not found)")
            continue

        print(f"Reading from: {input_path}")

        df = spark.read \
            .option("header", "true") \
            .schema(schemas[tag]) \
            .csv(input_path)

        df.repartition(4) \
          .write \
          .mode("overwrite") \
          .parquet(output_path)

        print(f"Written to: {output_path}")

In [31]:
write_to_parquet(12, year=2020)

Reading from: Data/raw/green/2020/01
Written to: Data/pq/green/2020/01
Reading from: Data/raw/green/2020/02
Written to: Data/pq/green/2020/02
Reading from: Data/raw/green/2020/03
Written to: Data/pq/green/2020/03
Reading from: Data/raw/green/2020/04
Written to: Data/pq/green/2020/04
Reading from: Data/raw/green/2020/05
Written to: Data/pq/green/2020/05
Reading from: Data/raw/green/2020/06
Written to: Data/pq/green/2020/06
Reading from: Data/raw/green/2020/07
Written to: Data/pq/green/2020/07
Reading from: Data/raw/green/2020/08
Written to: Data/pq/green/2020/08
Reading from: Data/raw/green/2020/09
Written to: Data/pq/green/2020/09
Reading from: Data/raw/green/2020/10
Written to: Data/pq/green/2020/10
Reading from: Data/raw/green/2020/11
Written to: Data/pq/green/2020/11
Reading from: Data/raw/green/2020/12
Written to: Data/pq/green/2020/12


In [32]:
write_to_parquet(12)

Reading from: Data/raw/green/2021/01
Written to: Data/pq/green/2021/01
Reading from: Data/raw/green/2021/02
Written to: Data/pq/green/2021/02
Reading from: Data/raw/green/2021/03
Written to: Data/pq/green/2021/03
Reading from: Data/raw/green/2021/04
Written to: Data/pq/green/2021/04
Reading from: Data/raw/green/2021/05
Written to: Data/pq/green/2021/05
Reading from: Data/raw/green/2021/06
Written to: Data/pq/green/2021/06
Reading from: Data/raw/green/2021/07
Written to: Data/pq/green/2021/07
Skipping Data/raw/green/2021/08 (not found)
Skipping Data/raw/green/2021/09 (not found)
Skipping Data/raw/green/2021/10 (not found)
Skipping Data/raw/green/2021/11 (not found)
Skipping Data/raw/green/2021/12 (not found)


In [33]:
write_to_parquet(12, tag="yellow", year=2020)

Reading from: Data/raw/yellow/2020/01
Written to: Data/pq/yellow/2020/01
Reading from: Data/raw/yellow/2020/02
Written to: Data/pq/yellow/2020/02
Reading from: Data/raw/yellow/2020/03
Written to: Data/pq/yellow/2020/03
Reading from: Data/raw/yellow/2020/04
Written to: Data/pq/yellow/2020/04
Reading from: Data/raw/yellow/2020/05
Written to: Data/pq/yellow/2020/05
Reading from: Data/raw/yellow/2020/06
Written to: Data/pq/yellow/2020/06
Reading from: Data/raw/yellow/2020/07
Written to: Data/pq/yellow/2020/07
Reading from: Data/raw/yellow/2020/08
Written to: Data/pq/yellow/2020/08
Reading from: Data/raw/yellow/2020/09
Written to: Data/pq/yellow/2020/09
Reading from: Data/raw/yellow/2020/10
Written to: Data/pq/yellow/2020/10
Reading from: Data/raw/yellow/2020/11
Written to: Data/pq/yellow/2020/11
Reading from: Data/raw/yellow/2020/12
Written to: Data/pq/yellow/2020/12


In [34]:
write_to_parquet(12, tag="yellow")

Reading from: Data/raw/yellow/2021/01
Written to: Data/pq/yellow/2021/01
Reading from: Data/raw/yellow/2021/02
Written to: Data/pq/yellow/2021/02
Reading from: Data/raw/yellow/2021/03
Written to: Data/pq/yellow/2021/03
Reading from: Data/raw/yellow/2021/04
Written to: Data/pq/yellow/2021/04
Reading from: Data/raw/yellow/2021/05
Written to: Data/pq/yellow/2021/05
Reading from: Data/raw/yellow/2021/06
Written to: Data/pq/yellow/2021/06
Reading from: Data/raw/yellow/2021/07
Written to: Data/pq/yellow/2021/07
Skipping Data/raw/yellow/2021/08 (not found)
Skipping Data/raw/yellow/2021/09 (not found)
Skipping Data/raw/yellow/2021/10 (not found)
Skipping Data/raw/yellow/2021/11 (not found)
Skipping Data/raw/yellow/2021/12 (not found)
